# EMail Ingest
The `frog-hollow.net` email domain is hosted on AWS Simple EMail Service (see https://github.com/froghollow/fh-ses), which triggers an AWS Lambda function to process and forward received emails.  

Lambda function [froghollow_ses_forwarder.py](https://github.com/froghollow/fh-ses/blob/main/modules/froghollow_email/froghollow_ses_forwarder.py) supports followup processing for particular senders specified in its PROCESS_BY_SENDER environment variable.  The send spec identifies each senders' processing module.  Currently, the only processor implemented is [`eml_to_markdown`](https://github.com/froghollow/fh-ses/blob/main/modules/froghollow_email/eml_to_markdown.py).  `email_to_markdown` extracts an email (.eml) message and stores its content in Amazon S3 as a Markdown document.

Functionality in this notebook retrieves content of `raw` Markdown files from Amazon S3, adds Obsidian properties, and stores `processed` files ready for wiki ingest. 



In [1]:
# initialize email ingestion process
import dotenv
dotenv.load_dotenv()

import os
s3_bucket = os.getenv("S3_BUCKET", "fh-danelfin-289755104220")
s3_folder = os.getenv("S3_FOLDER", "inbound/news")
WIKI_FOLDER = os.getenv("WIKI_FOLDER", "/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto")


import boto3
import re

s3_client = boto3.client("s3")

from datetime import datetime, timezone
today = datetime.now(timezone.utc)
today_str = today.strftime("%Y-%m-%d")



In [2]:
def list_emails_in_s3(s3_bucket: str, s3_folder: str, start_date: datetime, end_date: datetime) -> list[str]:
    """list email files in an S3 folder`

    Args:
        s3_bucket (str): The S3 bucket name.
        s3_folder (str): The folder in the S3 bucket.
        start_date (datetime): The start date for filtering emails.
        end_date (datetime): The end date for filtering emails.

    Returns:
        list[str]: The list of email file names and modification times in the folder between the specified start and end dates.
    """
    response = s3_client.list_objects_v2(Bucket=s3_bucket, Prefix=s3_folder)
    if "Contents" not in response:
        return []

    # filter objects by last modified date
    mod_date = lambda obj: obj["LastModified"]

    selected_objects = [] 
    for obj in response["Contents"]:
        # get modified date of the object converted to offset-naive datetime
        mod_date = obj["LastModified"] #.replace(tzinfo=None)
        if (start_date <= mod_date <= end_date):
            selected_objects.append(obj)

    return selected_objects

# start and end of today with tz
from datetime import datetime, timezone
today = datetime.now(timezone.utc)
start_of_today = datetime(today.year, today.month, today.day, tzinfo=timezone.utc)
end_of_today = datetime(today.year, today.month, today.day, 23, 59, 59, tzinfo=timezone.utc)

emails = list_emails_in_s3(s3_bucket, s3_folder, start_date=start_of_today, end_date=end_of_today)

for email in emails:
    print(email["Key"], email["LastModified"])

inbound/news/Hackers_Hunt_for_AI_Power_to_Steal_and_Resell.md 2026-11-11 12:44:15+00:00
inbound/news/The_10-Point_Rate-Hike_Fears_Return.md 2026-11-11 11:06:31+00:00
inbound/news/The_Morning_Download_AI_Is_Testing_the_Limits_of_Corporate_Governance.md 2026-11-11 14:45:38+00:00
inbound/news/The_Morning_Ledger_From_Good_to_Great.md 2026-11-11 12:14:06+00:00
inbound/news/WSJ_Politics_House_GOP_Saves_Hegseth_for_Now_and_Picks_Jordan_to_Lead.md 2026-11-11 12:05:54+00:00
inbound/news/_Hospital_Stocks_Are_Feeling_Chipper.md 2026-11-11 11:22:01+00:00


In [3]:
def retrieve_markdown_from_email(s3_bucket: str, s3_folder: str, email_filename: str) -> str:
    """retrieve markdown content from an email stored in S3

    Args:
        s3_bucket (str): The S3 bucket name.
        s3_folder (str): The folder in the S3 bucket.
        email_filename (str): The email file name.

    Returns:
        str: The markdown content of the email.
    """
    s3_key = f"{s3_folder}/{email_filename}"
    response = s3_client.get_object(Bucket=s3_bucket, Key=s3_key)
    # retrieve the email content as a string
    email_bytes = response["Body"].read()
    email_str = email_bytes.decode("utf-8")

    return email_str


In [4]:
def extract_field(content: str, field: str, verbose: bool = False) -> str: 
    """Extract a field from the markdown content.

    Args:
        content (str): The markdown content.
        field (str): The field to extract (e.g., "**From:**").

    Returns:
        str: A tuple containing the field and the line containing the field, or an empty string if not found.
    """
    index = content.find(field)
    if index != -1:
        line = content[index+len(field):].split("\n")[0].strip()
        if verbose:
            print(f"Extracted '{field}', '{line}'")
        return field, line
    
    if verbose:
        print(f"{field} not found")
    return field, "not found"


In [5]:
def process_email(email):     
    """Extracts properties from the given email and returns them as a dictionary.
    
    Arguments:
        email (dict): The Amazon S3 email object containing metadata and content.

    Returns:
        dict: A dictionary containing extracted properties from the email.
    
    """
    
    from datetime import datetime, timezone
    today = datetime.now(timezone.utc)
    today_str = today.strftime("%Y-%m-%d")

    email_filename = email["Key"].split("/")[-1]
    markdown_content = retrieve_markdown_from_email(s3_bucket, s3_folder, email_filename)
    #print(f"Markdown content for {email_filename}:\n{markdown_content}\n")
    print(f"Email last modified: {email['LastModified']}")

    source = extract_field(markdown_content, "**From:**")[1]
    author = source.split("<")[0].strip()

    properties = {
        "title": extract_field(markdown_content, "# ")[1],
        "source": source.replace('"',''),
        "author": source.split("<")[0].strip(),
        "published": extract_field(markdown_content, "**Date:**")[1][:10],
        "created": today_str,
        "description": "EMail content"
    }

    # string header for Obsidian properties 
    properties_str = "---\n"
    for p in properties:
        value = properties[p]
        properties_str += f"{p}: {value.replace(':','' )}\n"
    properties_str += "---\n"
    
    properties["output_filename"] = re.sub(r'[^a-zA-Z0-9\s]', '', properties["title"]).strip().replace(' ','_') + ".eml.md"
    properties["properties_str"] = properties_str
    properties["md"] = markdown_content

    return properties
    # break

"""import json 
for email in emails:
    properties = process_email(email)
    import json 
    print(json.dumps(properties, indent=4))
    print(today_str)
    break"""

'import json \nfor email in emails:\n    properties = process_email(email)\n    import json \n    print(json.dumps(properties, indent=4))\n    print(today_str)\n    break'

In [6]:
import json 

from datetime import datetime, timedelta, timezone
today = datetime.now(timezone.utc)
today_str = today.strftime("%Y-%m-%d")

# start and end of today with tz
# start = yesterday
yesterday = today - timedelta(days=1)
start_dt = datetime(yesterday.year, yesterday.month, yesterday.day, tzinfo=timezone.utc)
end_dt = datetime(today.year, today.month, today.day, 23, 59, 59, tzinfo=timezone.utc)
    
emails = list_emails_in_s3(s3_bucket, s3_folder, start_date=start_dt, end_date=end_dt)

for email in emails:
    print(email["Key"], email["LastModified"])
    properties = process_email(email)
    print(json.dumps(properties, indent=4))

    outfolder = f"{WIKI_FOLDER}/data/processed"
    outfile = f"{outfolder}/emails/{properties['published'].replace(' ','_')}/{properties['output_filename']}"
    properties["output_file"] = outfile
    os.makedirs(os.path.dirname(outfile), exist_ok=True)
    with open(outfile, "w") as f:
        f.write(properties['properties_str'] + properties['md'])
    #break




inbound/news/Hackers_Hunt_for_AI_Power_to_Steal_and_Resell.md 2026-11-11 12:44:15+00:00
Email last modified: 2026-11-11 12:44:15+00:00
{
    "title": "Hackers Hunt for AI Power to Steal and Resell",
    "source": "WSJ Pro Cybersecurity <access@interactive.wsj.com>",
    "author": "WSJ Pro Cybersecurity",
    "published": "2026-11-11",
    "created": "2026-11-11",
    "description": "EMail content",
    "output_filename": "Hackers_Hunt_for_AI_Power_to_Steal_and_Resell.eml.md",
    "properties_str": "---\ntitle: Hackers Hunt for AI Power to Steal and Resell\nsource: WSJ Pro Cybersecurity <access@interactive.wsj.com>\nauthor: WSJ Pro Cybersecurity\npublished: 2026-11-11\ncreated: 2026-11-11\ndescription: EMail content\n---\n",
    "md": "# Hackers Hunt for AI Power to Steal and Resell\n\n- **From:** WSJ Pro Cybersecurity <access@interactive.wsj.com>\n- **To:** richard@frog-hollow.net\n- **Date:** 2026-11-11T07:44:11-05:00\n\n---\n\nPlus: Spirit Airlines wins privacy concessions from Googl